# Project_R — working

End-to-end from the seven raw CSVs. Extract clock: **2026-06-30 23:59 IST**.

This notebook is the working. **Kernel → Run All.** It executes `01_data_audit.py` … `09_deliverables.py` in order (the same files as `./run_all.sh`), then shows the locked headlines, waterfall, and regression check.

Do not copy code out of the scripts into a second place — the `.py` files are the source of truth; this notebook is the single Run-All entry point so a reviewer does not have to stitch them by hand.

| Submit | File |
|---|---|
| Memo | `MEMO.docx` / `MEMO.md` |
| Deck | `DECK.pptx` |
| Working | this notebook, or `./run_all.sh` |

Optional live sliders: `python3 -m streamlit run sensitivity_explorer.py`


## How to run

```bash
python3 -m pip install -r requirements.txt
jupyter notebook Project_R.ipynb    # or open in VS Code / JupyterLab
# Kernel → Run All
```

The seven CSVs must sit in the **same folder** as this notebook: `captains.csv`, `doc_events.csv`, `approvals.csv`, `activation.csv`, `nudges.csv`, `airport_hourly.csv`, `airport_trips.csv`.

Equivalent without Jupyter: `./run_all.sh` (writes the same memo/deck/waterfall and fails if headlines moved).


## 0. Setup


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "captains.csv").exists():
    raise SystemExit(
        f"CSVs not found in {ROOT}. Open the notebook from the repo root "
        "(the folder that contains captains.csv)."
    )

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")]
)
print("Python", sys.version.split()[0])
print("ROOT", ROOT)


## 1. Execute the full working (merged steps)

Each cell runs one script with `runpy` so `__name__ == "__main__"` fires. Same prints as the `0N_*_output.txt` logs. Shared maths stay in `metrics.py`.


In [ ]:
import runpy

def run_step(name: str) -> None:
    print("=" * 88)
    print(name)
    print("=" * 88)
    runpy.run_path(str(ROOT / name), run_name="__main__")


### Step 1 — data audit (schema, joins, mature flags)


In [ ]:
run_step("01_data_audit.py")


### Step 2 — A1 funnel (empirical ~15.6-day cut, n = 21,024)


In [ ]:
run_step("02_funnel.py")


### Step 3 — where volume is lost


In [ ]:
run_step("03_dropoff.py")


### Step 4 — channel lens; C1 capture-only 1,637 / 411


In [ ]:
run_step("04_channel_leaks.py")


### Step 5 — A3 CAMP_WA_002 (do not scale 5×)


In [ ]:
run_step("05_campaign.py")


### Step 6 — B1 airport hourly mismatch


In [ ]:
run_step("06_airport_hourly.py")


### Step 7 — B2 post-trip economics


In [ ]:
run_step("07_airport_trips.py")


### Step 8 — C1a / C1b / derived ARA ₹35.4


In [ ]:
run_step("08_intervention_sizing.py")


### Step 9 — waterfall, MEMO.docx, DECK.pptx


In [ ]:
run_step("09_deliverables.py")


## 2. Locked headlines (same functions as Streamlit / `check_regression.py`)

Bankable: C1a + C1b ≈ 180/month (disjoint). ARA is ₹/leg, not added to 180. Do not bank fos 128–256. Do not hire the airport catchment.


In [ ]:
from IPython.display import Image, display
import pandas as pd

from metrics import (
    C1A_CENTRAL_SHOWUP,
    airport_hourly_snapshot,
    ara_economics,
    ara_monthly_cost,
    c1a_monthly,
    c1b_monthly,
    event_funnel,
    headlines,
    mature_months,
)
import check_regression

h = headlines()
eco = ara_economics()
hourly = airport_hourly_snapshot()
months, span_days, tmin, tmax = mature_months()
funnel = event_funnel()
n = len(funnel)
n_appr = int(funnel["final_status"].eq("approved").sum())

summary = pd.DataFrame(
    [
        ["Mature ∩ has doc_events", f"{h['mature_has_events_n']:,}"],
        ["Approved | that cohort", f"{n_appr:,} / {n:,} = {100 * n_appr / n:.2f}%"],
        ["Mature window", f"{span_days} days ≈ {months:.3f} months"],
        ["RC capture-only (C1a stock)", f"{h['rc_capture_only_n']:,}"],
        ["Insurance capture-only (C1b stock)", f"{h['insurance_capture_only_n']:,}"],
        ["C1a / month @ 60% show-up × RC att-3", f"{c1a_monthly(C1A_CENTRAL_SHOWUP):.1f}"],
        ["C1b / month @ att-2", f"{c1b_monthly(2):.1f}"],
        ["C1a + C1b att-2 (disjoint)", f"{c1a_monthly(C1A_CENTRAL_SHOWUP) + c1b_monthly(2):.1f}"],
        ["Airport unfulfilled vs elsewhere", f"{100 * hourly['airport_unf_share']:.0f}% vs {100 * hourly['other_unf_share']:.0f}%"],
        ["Share of airport unfulfilled in 21:00–03:59", f"{100 * hourly['night_share_of_airport_unf']:.0f}%"],
        ["Captains online night vs rest of day", f"{hourly['mean_captains_worst']:.0f} vs {hourly['mean_captains_rest']:.0f}"],
        ["ARA ₹ / eligible night-suburban leg", f"₹{eco['payout_per_eligible_leg']:.2f}"],
        ["ARA sample ₹ / month @ 100% of derived", f"₹{ara_monthly_cost(1.0):,.0f}"],
    ],
    columns=["item", "value"],
)
display(summary)

print("\nRegression lock:")
rc = check_regression.main()
if rc != 0:
    raise SystemExit("Headline regression failed")


In [ ]:
wf = ROOT / "figures" / "funnel_waterfall.png"
if wf.exists():
    display(Image(filename=str(wf)))
else:
    print("waterfall missing — step 9 should have written it")


## Standing rules (do not re-litigate in the debrief)

- Mature = signup age **>** max `in_progress` age (**15.602778** days). Funnel = mature ∩ has `doc_events`.
- `verification_pass` in `doc_events` is source of truth for documents. Rejected (409) is a post-clearance gate.
- Zero-doc mature (1,416) = never-attempt, reported separately.
- 457 nudges `clicked=1 & delivered=0` are dropped. CAMP_WA_002 clicked vs not ≈ **0 pp**; do not scale 5×.
- `airport_hourly.csv` and `airport_trips.csv` do not mix. `signup_zone_id` does not join airport zones. Trips are **sampled**.
- C1a show-up is unobserved (central 60%). ARA ₹/month is sample-implied, not a city P&L. 30% of fare overpays vs ₹35.4/leg.
